In [746]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.autograd as autograd
import torch.optim as optim
import torch.nn.functional as F

In [747]:
!pip install scikit-learn

In [748]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if(torch.cuda.is_available()):
    print(f"GPU available: {torch.cuda.get_device_name()}")
    gpu = True
else:
    print("No GPU available")

GPU available: NVIDIA GeForce RTX 5050 Laptop GPU


In [749]:
matches_path = "data/Matches.csv"

df = pd.read_csv(matches_path)
print(df.shape)

(230557, 48)


/tmp/ipykernel_1057341/4087795544.py:3: DtypeWarning: Columns (0: MatchTime) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(matches_path)


In [750]:
features_to_drop = [
    'HTHome', 'HTAway', 'HTResult', 
    'HomeFouls', 'AwayFouls', 'HomeCorners', 'AwayCorners', 
    'HomeYellow', 'AwayYellow', 'HomeRed', 'AwayRed',    
    'OddHome', 'OddDraw', 'OddAway', 'MaxHome', 'MaxDraw', 'MaxAway', 
    'Over25', 'Under25', 'MaxOver25', 'MaxUnder25', 
    'HandiSize', 'HandiHome', 'HandiAway', 
    'C_LTH', 'C_LTA', 'C_VHD', 'C_VAD', 'C_HTB', 'C_PHB',    
]

thin_df = df.drop(features_to_drop, axis="columns")
thin_df = thin_df.dropna()
print(thin_df.shape)
thin_df.head()

(31463, 18)


,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway,FTResult,HomeShots,AwayShots,HomeTarget,AwayTarget
162577,F2,2019-07-26,19:00:00,Ajaccio,Le Havre,1360.18,1438.67,4.0,4.0,1.0,4.0,2.0,2.0,D,12.0,12.0,4.0,3.0
162578,F2,2019-07-26,19:00:00,Chambly,Valenciennes,1315.69,1361.10,0.0,0.0,4.0,5.0,1.0,0.0,H,14.0,9.0,5.0,0.0
162579,F2,2019-07-26,19:00:00,Clermont,Chateauroux,1414.77,1396.05,1.0,3.0,7.0,11.0,3.0,0.0,H,11.0,11.0,7.0,5.0
162580,F2,2019-07-26,19:00:00,Guingamp,Grenoble,1471.61,1364.04,2.0,3.0,1.0,5.0,3.0,3.0,D,22.0,5.0,8.0,3.0
162581,F2,2019-07-26,19:00:00,Nancy,Orleans,1364.51,1380.99,3.0,9.0,1.0,1.0,0.0,0.0,D,8.0,6.0,1.0,3.0


In [751]:
for i in range(5):
    print(f"Results format: {thin_df.iloc[i, 11]}") # home goals
    print(f"Results format: {thin_df.iloc[i, 12]}") # away goals
    print(f"Results format: {thin_df.iloc[i, 13]}") # 13 feature is the result
    

Results format: 2.0
Results format: 2.0
Results format: D
Results format: 1.0
Results format: 0.0
Results format: H
Results format: 3.0
Results format: 0.0
Results format: H
Results format: 3.0
Results format: 3.0
Results format: D
Results format: 0.0
Results format: 0.0
Results format: D


In [752]:
def compute_avg(df, group_ft, feature, n = 5):
    """
        INPUT:
            df: dataframe,
            group_ft: feature that groups the samples for the avg,
            feature: feature to compute the avarage of
            n: consecutive values to consider for the avg
    """
    avg = df.groupby(group_ft)[feature].shift(1).rolling(window=5, min_periods=1).mean()
    return avg

df_home = thin_df[["MatchDate", "HomeTeam", "HomeShots", "FTHome", "HomeTarget"]].copy()
df_home = df_home.rename(columns={
    'HomeTeam': 'team',
    'FTHome': 'goals',
    'HomeShots': 'shots',
    'HomeTarget': 'target'
})

df_away = thin_df[["MatchDate", "AwayTeam", "AwayShots", "FTAway", "AwayTarget"]].copy()    
df_away = df_away.rename(columns={
    'AwayTeam': 'team',
    'FTAway': 'goals',
    'AwayShots': 'shots',
    'AwayTarget': 'target'
})

cat_df = pd.concat([df_home, df_away])

# at this point in cat_df we have the list matches for teams grouped by and sorted
cat_df = cat_df.sort_values(by=['team', 'MatchDate'])
cat_df.info()

<class 'pandas.DataFrame'>
Index: 62926 entries, 172290 to 230426
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   MatchDate  62926 non-null  str    
 1   team       62926 non-null  str    
 2   shots      62926 non-null  float64
 3   goals      62926 non-null  float64
 4   target     62926 non-null  float64
dtypes: float64(3), str(2)
memory usage: 2.9 MB


In [753]:
cat_df['Form5Shots'] = compute_avg(cat_df, 'team', 'shots')
cat_df['Form5Target'] = compute_avg(cat_df, 'team', 'target')
cat_df['Form5Goals'] = compute_avg(cat_df, 'team', 'goals')

thin_df = thin_df.merge(
    cat_df,
    left_on=['MatchDate', 'HomeTeam'],
    right_on=['MatchDate', 'team'],
    how='left'
)

thin_df = thin_df.rename(columns={
    'Form5Shots': 'Form5ShotsHome',
    'Form5Target': 'Form5TargetHome',
    'Form5Goals': 'Form5GoalsHome'
    })
thin_df = thin_df.drop(columns=['team', 'goals', 'shots', 'target'])

thin_df = thin_df.merge(
    cat_df,
    left_on=['MatchDate', 'AwayTeam'],
    right_on=['MatchDate', 'team'],
    how='left'
)

thin_df = thin_df.rename(columns={
    'Form5Shots': 'Form5ShotsAway',
    'Form5Target': 'Form5TargetAway',
    'Form5Goals': 'Form5GoalsAway'
})
thin_df = thin_df.drop(columns=['team', 'goals', 'shots', 'target'])
thin_df = thin_df.drop(columns=['HomeShots', 'AwayShots', 'HomeTarget', 'AwayTarget', 'FTHome', 'FTAway'])

In [754]:
df = thin_df.dropna()
df.info()

<class 'pandas.DataFrame'>
Index: 31462 entries, 0 to 31462
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Division         31462 non-null  str    
 1   MatchDate        31462 non-null  str    
 2   MatchTime        31462 non-null  str    
 3   HomeTeam         31462 non-null  str    
 4   AwayTeam         31462 non-null  str    
 5   HomeElo          31462 non-null  float64
 6   AwayElo          31462 non-null  float64
 7   Form3Home        31462 non-null  float64
 8   Form5Home        31462 non-null  float64
 9   Form3Away        31462 non-null  float64
 10  Form5Away        31462 non-null  float64
 11  FTResult         31462 non-null  str    
 12  Form5ShotsHome   31462 non-null  float64
 13  Form5TargetHome  31462 non-null  float64
 14  Form5GoalsHome   31462 non-null  float64
 15  Form5ShotsAway   31462 non-null  float64
 16  Form5TargetAway  31462 non-null  float64
 17  Form5GoalsAway   31462 non-n

In [755]:
# make the 'date' and 'time' feature as int values that we store in our dataframe.
date_parts = df.iloc[:, 1].str.split("-", expand=True)

df['Year'] = date_parts[0].astype('int16')
df['Month'] = date_parts[1].astype('int16')
df['Day'] = date_parts[2].astype('int16')

time_parts = df.iloc[:, 2].str.split(":", expand=True)

df['Hour'] = time_parts[0].astype('int8')
df['Minutes'] = time_parts[1].astype('int8')

# in the end we drop the previous features stored as strings.
df_min = df.drop(columns=["MatchDate", "MatchTime", "HomeTeam", "AwayTeam", "Division"])

In [756]:
# define an array of unique team names' string.
unique_teams = [x for x in df['HomeTeam'].unique()]

print(unique_teams)
print(len(unique_teams))

['Ajaccio', 'Chambly', 'Clermont', 'Guingamp', 'Nancy', 'Niort', 'Rodez', 'Sochaux', 'Genk', 'Stuttgart', 'Dresden', 'Le Mans', 'Holstein Kiel', 'Osnabruck', 'Cercle Brugge', 'St Truiden', 'Waregem', 'Hamburg', 'Anderlecht', 'Greuther Furth', 'Regensburg', 'Wehen', 'Charleroi', 'Eupen', 'Bielefeld', 'Lorient', 'Bochum', 'Sandhausen', 'Auxerre', 'Chateauroux', 'Grenoble', 'Le Havre', 'Orleans', 'Paris FC', 'Troyes', 'Valenciennes', 'Zwolle', 'St Pauli', 'Club Brugge', 'Luton', 'Karlsruhe', 'Hannover', 'Celtic', 'Hibernian', 'Livingston', 'Ross County', 'Barnsley', 'Blackburn', 'Brentford', 'Millwall', 'Reading', 'Stoke', 'Swansea', 'Wigan', 'Lens', 'Standard', 'FC Emmen', 'Vitesse', 'Nottm Forest', 'Kortrijk', 'Oostende', 'Mechelen', 'Twente', 'VVV Venlo', 'Heracles', 'Heidenheim', 'Kilmarnock', 'Gent', 'Feyenoord', 'Darmstadt', 'Erzgebirge Aue', 'Aberdeen', 'Den Haag', 'Bristol City', 'Mouscron', 'AZ Alkmaar', 'Nurnberg', 'Huddersfield', 'Caen', 'Sparta Rotterdam', 'Monaco', 'Liverpool

In [757]:
# code taken from the https://gist.github.com/GavinXing/9954ea846072e115bb07d9758892382c
# so far we are not using it since for simplicity we drop the TeamNames columns from the dataset

class CBOW(nn.Module):

    def __init__(self, context_size=2, embedding_size=100, vocab_size=None):
        super(CBOW, self).__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_size)
        self.linear1 = nn.Linear(embedding_size, vocab_size)

    def forward(self, inputs):
        lookup_embeds = self.embeddings(inputs)
        embeds = lookup_embeds.sum(dim=0)
        out = self.linear1(embeds)
        out = F.log_softmax(out)
        return out
    
def make_context_vector(context, word_to_ix):
    idxs = [word_to_ix[w] for w in context]
    tensor = torch.LongTensor(idxs)
    return autograd.Variable(tensor)

In [758]:
vocab = set(unique_teams)
vocab_size = len(vocab)

word_to_ix = {word: i for i, word in enumerate(vocab)}
data = []

In [759]:
result_mapping = {'H': 0, 'D': 1, 'A': 2}
df_min['Target'] = df_min['FTResult'].map(result_mapping).astype('int8')

df_min = df_min.drop(columns=["FTResult"])
df_min.info()

<class 'pandas.DataFrame'>
Index: 31462 entries, 0 to 31462
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   HomeElo          31462 non-null  float64
 1   AwayElo          31462 non-null  float64
 2   Form3Home        31462 non-null  float64
 3   Form5Home        31462 non-null  float64
 4   Form3Away        31462 non-null  float64
 5   Form5Away        31462 non-null  float64
 6   Form5ShotsHome   31462 non-null  float64
 7   Form5TargetHome  31462 non-null  float64
 8   Form5GoalsHome   31462 non-null  float64
 9   Form5ShotsAway   31462 non-null  float64
 10  Form5TargetAway  31462 non-null  float64
 11  Form5GoalsAway   31462 non-null  float64
 12  Year             31462 non-null  int16  
 13  Month            31462 non-null  int16  
 14  Day              31462 non-null  int16  
 15  Hour             31462 non-null  int8   
 16  Minutes          31462 non-null  int8   
 17  Target           31462 non-n

In [760]:
from torch.utils.data import TensorDataset, DataLoader

y_labels = df_min.iloc[:, 17]
X_features = df_min.drop(df_min.columns[17], axis=1)

In [761]:
from sklearn.model_selection import train_test_split
X, X_test, y, y_test = train_test_split(
    X_features, y_labels, test_size=0.20, random_state=42, shuffle=False
)

X_train, x_eval, y_train, y_eval = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False
)

print(X_train.shape)
print(x_eval.shape)
print(X_test.shape)

(20135, 17)
(5034, 17)
(6293, 17)


In [762]:
X_train = torch.tensor(X_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)

X_test = torch.tensor(X_test.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.long)

X_eval = torch.tensor(x_eval.values, dtype=torch.float32)
y_eval = torch.tensor(y_eval.values, dtype=torch.long)

train_dataloader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=False)
test_dataloader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)
eval_dataloader = DataLoader(TensorDataset(X_eval, y_eval), batch_size=64, shuffle=False)


In [763]:
class ModelClassifier(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        
        self.lin1 = nn.Linear(num_features, out_features=32) # linear applies x * W^T + b
        self.lin2 = nn.Linear(32, out_features=16)
        self.output_layer = nn.Linear(16, out_features=3) # output layer
        
        self.hidden_act = nn.ReLU()
        # self.output_act = nn.Softmax() # defines the SoftMax behaviour for more than 2 classes classification
        self.dropout = nn.Dropout(p=0.3)
        
    def forward(self, x):        
        # activation of the first layer
        x = self.lin1(x)
        x = self.hidden_act(x)
        x = self.dropout(x)
        
        # activation of the second layer
        x = self.lin2(x)
        x = self.hidden_act(x)
        x = self.dropout(x)
        
        x = self.output_layer(x)        
        return x

Define some usefull elements for the training like:
- Initialize the model;
- Initialize hyperparameters (learning rate, weight decay);
- Initialize Optimizers;

In [764]:
num_features = X_train.shape[1]

EPOCHS = 30
EPOCHS_STOP = 5

lr = 1e-02
weight_decay = 0

model = ModelClassifier(num_features=num_features)
criterion = nn.CrossEntropyLoss()
model.to(device)

optimizer = optim.SGD(
    model.parameters(),
    lr=lr,
    weight_decay=0,
)
lr_scheduler = optim.lr_scheduler.MultiplicativeLR(
    optimizer=optimizer,
    lr_lambda= lambda epoch: 0.95
)

### EVAL FUNCTION 

In [765]:
def eval_model(model, dataloader, loss_fn, device):
    """
        Function that validates the model
        INPUTS:
            - model: our defined class model;
            - dataloader: an instance of the validation dataloader containing samples and labels;
            - loss_fn: loss function (just for health checking the results);
            - device: gpu or cpu boolean value.
    """
    
    model.eval()  # Set to evaluation mode

    total_loss = 0
    correct = 0
    samples = 0
    metrics = {} # further implementation
    
    with torch.no_grad():
        for input_batch, label_batch in dataloader:
            input_batch = input_batch.to(device)
            label_batch = label_batch.to(device)
            
            label_batch = label_batch.long()
            
            predict = model(input_batch)
            
            loss = loss_fn(predict, label_batch)
            
            total_loss += loss.item()
            predictions = torch.argmax(predict, dim=1)
            correct += (predictions == label_batch).sum().item() # add up the number of correct training predictions
            samples += label_batch.size(0) # Add batch size (e.g., 32)
                
    model.train()  # Set back to training mode
    metrics["Avg_loss"] = total_loss / len(dataloader)
    metrics["Accuracy"] = correct / samples
    print("Number of samples:", samples)
    
    return metrics

### TRAINING

In [766]:
#model = ModelClassifier(num_features=num_features)
#model.to(device)
#model.train(True)

train_loss = []
eval_loss = []

model.train()

# set these values for the early stopping
best_loss = float('inf')
count = 0
for epoch in range(EPOCHS):
    total_loss = 0
    correct = 0
    for input_batch, label_batch in train_dataloader:
        input_batch = input_batch.to(device)
        label_batch = label_batch.to(device)
        
        label_batch = label_batch.long()
        
        logits = model(input_batch)
        loss = criterion(logits, label_batch)
        
        # zero-out the gradients
        optimizer.zero_grad()
        loss.backward() # perform backpropagation
        
        # compute the next step for the optimizer and the lr_scheduler
        optimizer.step()
        lr_scheduler.step()
        
        # compute total training loss and correct predictions
        total_loss += loss.item()
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == label_batch).sum().item() # add up the number of correct training predictions
                
    metrics = eval_model(model, eval_dataloader, criterion, device=device)
    
    if epoch % 3 == 0:
        print(f"Loss for epoch nr. {epoch + 1}: {total_loss:.2f}")
        print(f"Correct predictions for epoch nr. {epoch + 1}: {correct}\n")
        
        #for logit, label in zip(logits, label_batch):
        #    prediction = F.softmax(logit, dim=0)
        #    print(f"Our prediction: {prediction}")
        #    print(f"True prediction: {label}")
        print(f"Eval Loss for epoch nr. {epoch + 1}: {metrics["Avg_loss"]:.2f}")
        print(f"Eval percentage of correct predictions for epoch nr. {epoch + 1}: {metrics["Accuracy"]*100:.2f}\n")
    
    train_loss.append(total_loss) # append loss relative to the epoch
    eval_loss.append(metrics["Avg_loss"])
        
    if total_loss < best_loss:
        best_loss = total_loss
        count = 0 # reset the count because we improved the loss
    else:
        count += 1
        
    if count == EPOCHS_STOP:
        print(f"Early stopping at epoch nr. {epoch + 1}")
        break # we stop the training if we have no improvement   
    
mean_train_loss = np.mean(train_loss)
mean_eval_loss = np.mean(eval_loss)
print(f"Mean train loss: {mean_train_loss:.4f}")
print(f"Mean eval loss: {mean_eval_loss:.4f}")
    

Number of samples: 5034
Loss for epoch nr. 1: 762.58
Correct predictions for epoch nr. 1: 6869

Number of samples: 5034
Number of samples: 5034
Number of samples: 5034
Loss for epoch nr. 4: 343.21
Correct predictions for epoch nr. 4: 6793

Number of samples: 5034
Number of samples: 5034
Number of samples: 5034
Loss for epoch nr. 7: 343.11
Correct predictions for epoch nr. 7: 6817

Number of samples: 5034
Number of samples: 5034
Number of samples: 5034
Loss for epoch nr. 10: 343.27
Correct predictions for epoch nr. 10: 6741

Number of samples: 5034
Number of samples: 5034
Number of samples: 5034
Loss for epoch nr. 13: 343.19
Correct predictions for epoch nr. 13: 6737

Number of samples: 5034
Number of samples: 5034
Number of samples: 5034
Loss for epoch nr. 16: 343.14
Correct predictions for epoch nr. 16: 6814

Number of samples: 5034
Number of samples: 5034
Number of samples: 5034
Loss for epoch nr. 19: 343.07
Correct predictions for epoch nr. 19: 6867

Early stopping at epoch nr. 19
M